Testing

In [1]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
!pip install dotnet.loadenv

ERROR: Could not find a version that satisfies the requirement dotnet.loadenv (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for dotnet.loadenv


In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI()

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "query_database",
            "description": "Execute a read only SQL query against the InsightAgent business database and return the results.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "A read only SQL query to execute against the database."
                    }
                },
                "required": ["sql"]
            }
        }
    }
]

In [6]:
messages = [
    {
        "role": "system",
        "content": """
You are a business data analyst.

You have access to a database containing ecommerce business data.

When you need factual information from the database, use the query_database tool.
Do not invent database results.
"""
    },
    {
        "role": "user",
        "content": "How many orders are in the database?"
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

response.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Dma7hwdNaOpbkC1C1NjJ2nLQ', function=Function(arguments='{"sql":"SELECT COUNT(*) AS total_orders FROM orders;"}', name='query_database'), type='function')])

In [7]:
response.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Dma7hwdNaOpbkC1C1NjJ2nLQ', function=Function(arguments='{"sql":"SELECT COUNT(*) AS total_orders FROM orders;"}', name='query_database'), type='function')])

In [ ]:
import json 

tool_call = response.choices[0].message.tool_calls[0]

tool_name = tool_call.function.name
tool_arguments = json.loads(tool_call.function.arguments)

print("Tool:", tool_name)
print("Arguments:", tool_arguments)

Tool: query_database
Arguments: {'sql': 'SELECT COUNT(*) AS total_orders FROM orders;'}


In [ ]:
from src.tools.database_tools import query_database


PROJECT_ROOT = Path(__file__).resolve().parents[2]

DATABASE_PATH = (
    PROJECT_ROOT
    / "data"
    / "database"
    / "insight_agent.duckdb"
)


tool_result = query_database(**tool_arguments)
tool_result

IOException: IO Error: Cannot open file "C:\Users\DELL\Projects\insight-agent\notebooks\data\database\insight_agent.duckdb": The system cannot find the path specified.
